# Download DermaXplain Datasets from Google Drive

This notebook downloads the prepared dataset ZIP files from Google Drive using `gdown`, extracts them, and verifies the expected directory structure.

Expected final structure:

```text
data/
|--ham10000/
│   |-- images/
│   |-- masks/
│   |-- HAM10000_metadata.csv
|-- isic2018/
    |-- images/
    |-- masks/
```

Before running this notebook, make sure both Google Drive ZIP files are shared as:

```text
Anyone with the link -> Viewer
```


In [1]:
from pathlib import Path
import sys
import subprocess
import zipfile

# Detect project root.
# If this notebook is run from src/, PROJECT_ROOT becomes the parent folder.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "src":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
ARCHIVE_DIR = PROJECT_ROOT / "data_archives"

DATA_DIR.mkdir(exist_ok=True)
ARCHIVE_DIR.mkdir(exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Data dir:    ", DATA_DIR)
print("Archive dir: ", ARCHIVE_DIR)


Project root: /home/jessica/Projects/DCU/2026-mcm-XAI-skin-lesion
Data dir:     /home/jessica/Projects/DCU/2026-mcm-XAI-skin-lesion/data
Archive dir:  /home/jessica/Projects/DCU/2026-mcm-XAI-skin-lesion/data_archives


## 1. Install `gdown` if needed

In [2]:
try:
    import gdown
    print("gdown already installed")
except ImportError:
    print("Installing gdown...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "gdown"])
    import gdown
    print("gdown installed")


Installing gdown...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [gdown]32m1/2 [gdown]
gdown installed


## 2. Add Google Drive links

Replace the placeholder values below with the Google Drive share links for:

- `ham10000.zip`
- `isic2018.zip`


In [3]:
HAM10000_URL = "https://drive.google.com/file/d/1HZGCoyB3K67CToa-iCGwqAXE4dV-gewN/view?usp=sharing"
ISIC2018_URL = "https://drive.google.com/file/d/1y31iCPOMvNk_WcJIcCtM-76NeTE_SLHo/view?usp=sharing"

HAM10000_ZIP = ARCHIVE_DIR / "ham10000.zip"
ISIC2018_ZIP = ARCHIVE_DIR / "isic2018.zip"

print("HAM10000 target:", HAM10000_ZIP)
print("ISIC2018 target:", ISIC2018_ZIP)


HAM10000 target: /home/jessica/Projects/DCU/2026-mcm-XAI-skin-lesion/data_archives/ham10000.zip
ISIC2018 target: /home/jessica/Projects/DCU/2026-mcm-XAI-skin-lesion/data_archives/isic2018.zip


## 3. Download archives

In [4]:
import gdown
from pathlib import Path

HAM10000_FILE_ID = "1HZGCoyB3K67CToa-iCGwqAXE4dV-gewN"
ISIC2018_FILE_ID = "1y31iCPOMvNk_WcJIcCtM-76NeTE_SLHo"

HAM10000_ZIP = ARCHIVE_DIR / "ham10000.zip"
ISIC2018_ZIP = ARCHIVE_DIR / "isic2018.zip"

def download_if_missing(file_id: str, output_path: Path) -> None:
    """Download a Google Drive file with gdown unless it already exists."""
    if output_path.exists():
        print(f"Already exists, skipping: {output_path}")
        return

    url = f"https://drive.google.com/uc?id={file_id}"

    print(f"Downloading to: {output_path}")
    result = gdown.download(url, str(output_path), quiet=False)

    if result is None or not output_path.exists():
        raise RuntimeError(f"Download failed for {output_path.name}")

download_if_missing(HAM10000_FILE_ID, HAM10000_ZIP)
download_if_missing(ISIC2018_FILE_ID, ISIC2018_ZIP)


Downloading...
From (original): https://drive.google.com/uc?id=1HZGCoyB3K67CToa-iCGwqAXE4dV-gewN
From (redirected): https://drive.google.com/uc?id=1HZGCoyB3K67CToa-iCGwqAXE4dV-gewN&confirm=t&uuid=5755abb0-99fe-4f6c-91d4-298bd787ab26
To: /home/jessica/Projects/DCU/2026-mcm-XAI-skin-lesion/data_archives/ham10000.zip
100%|██████████| 2.78G/2.78G [04:08<00:00, 11.2MB/s]


Downloading...
From (original): https://drive.google.com/uc?id=1y31iCPOMvNk_WcJIcCtM-76NeTE_SLHo
From (redirected): https://drive.google.com/uc?id=1y31iCPOMvNk_WcJIcCtM-76NeTE_SLHo&confirm=t&uuid=64a2fd0f-9f12-403c-89f4-d2d1cfbe537e
To: /home/jessica/Projects/DCU/2026-mcm-XAI-skin-lesion/data_archives/isic2018.zip
100%|██████████| 11.2G/11.2G [17:11<00:00, 10.8MB/s]  


## 4. Check archive sizes

In [5]:
def size_gb(path: Path) -> float:
    return path.stat().st_size / (1024 ** 3)

for archive in [HAM10000_ZIP, ISIC2018_ZIP]:
    if archive.exists():
        print(f"{archive.name}: {size_gb(archive):.2f} GB")
    else:
        print(f"Missing: {archive}")


ham10000.zip: 2.59 GB
isic2018.zip: 10.41 GB


## 5. Extract archives

The ZIPs were created from the project root, so they should contain paths such as:

```text
data/ham10000/...
data/isic2018/...
```

Therefore, they are extracted into `PROJECT_ROOT`.


In [6]:
def extract_if_needed(zip_path: Path, expected_dir: Path) -> None:
    """Extract ZIP into PROJECT_ROOT unless the expected folder already exists."""
    if expected_dir.exists():
        print(f"Already extracted, skipping: {expected_dir}")
        return

    if not zip_path.exists():
        raise FileNotFoundError(f"Missing archive: {zip_path}")

    print(f"Extracting: {zip_path}")
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(PROJECT_ROOT)
    print(f"Extracted: {zip_path.name}")

extract_if_needed(HAM10000_ZIP, DATA_DIR / "ham10000" / "images")
extract_if_needed(ISIC2018_ZIP, DATA_DIR / "isic2018" / "images")


Extracting: /home/jessica/Projects/DCU/2026-mcm-XAI-skin-lesion/data_archives/ham10000.zip
Extracted: ham10000.zip
Extracting: /home/jessica/Projects/DCU/2026-mcm-XAI-skin-lesion/data_archives/isic2018.zip
Extracted: isic2018.zip


## 6. Verify expected structure

In [7]:
required_paths = [
    DATA_DIR / "ham10000" / "images",
    DATA_DIR / "ham10000" / "masks",
    DATA_DIR / "isic2018" / "images",
    DATA_DIR / "isic2018" / "masks",
]

all_ok = True

for path in required_paths:
    exists = path.exists()
    print(f"{path}: {'OK' if exists else 'MISSING'}")
    all_ok = all_ok and exists

if not all_ok:
    raise FileNotFoundError("One or more required dataset folders are missing.")

print("Dataset structure OK")


/home/jessica/Projects/DCU/2026-mcm-XAI-skin-lesion/data/ham10000/images: OK
/home/jessica/Projects/DCU/2026-mcm-XAI-skin-lesion/data/ham10000/masks: OK
/home/jessica/Projects/DCU/2026-mcm-XAI-skin-lesion/data/isic2018/images: OK
/home/jessica/Projects/DCU/2026-mcm-XAI-skin-lesion/data/isic2018/masks: OK
Dataset structure OK


## 7. Count files

In [8]:
def count_files(path: Path) -> int:
    return sum(1 for p in path.rglob("*") if p.is_file())

counts = {
    "HAM10000 images": count_files(DATA_DIR / "ham10000" / "images"),
    "HAM10000 masks": count_files(DATA_DIR / "ham10000" / "masks"),
    "ISIC2018 images": count_files(DATA_DIR / "isic2018" / "images"),
    "ISIC2018 masks": count_files(DATA_DIR / "isic2018" / "masks"),
}

for name, count in counts.items():
    print(f"{name}: {count}")


HAM10000 images: 10015
HAM10000 masks: 10015
ISIC2018 images: 2594
ISIC2018 masks: 2594


## 8. Final check

In [ ]:
print("Dataset setup complete.")
print()
print("You can now run:")
print("- 01_data_inventory.ipynb")
print("- 01_data_preprocessing.ipynb")
print("- 03_create_pilot_subset.ipynb")


Dataset setup complete.

You can now run:
- 01_data_inventory.ipynb
- 01_data_preprocessing.ipynb
- 03_create_pilot_subset.ipynb


: 